# LeetCode 228 — Summary Ranges bằng Python

- Link đề: https://leetcode.com/problems/summary-ranges/
- Độ khó: **Easy** · Chủ đề: Array, duyệt tuần tự, two pointers

Chữ ký hàm LeetCode cần viết:

```python
class Solution:
    def summaryRanges(self, nums: list[int]) -> list[str]:

```

**Mục tiêu học của notebook**

1. Hiểu đúng thế nào là một "đoạn liên tiếp" (range) trong mảng đã sắp xếp, không trùng.
2. Ôn cú pháp Python cần dùng: list, index, `len`, `append`, f-string, vòng `while` lồng `while`.
3. Đi từ cách trực giác (kiểm tra từng số có thuộc đoạn trước không) tới cách **hai con trỏ `start`/`end`** gọn gàng.
4. Biết thêm một cách "khéo": nhóm theo hiệu `nums[i] - i` với `itertools.groupby`.
5. Tự trace, tự test, nhận ra lỗi hay gặp (quên đoạn cuối, mảng rỗng, tràn khi `+1`) và giải bài biến thể.

## 1. Đề bài và ví dụ

Cho mảng số nguyên `nums` **đã sắp xếp tăng dần** và **không có phần tử trùng**.

Một *range* `[a, b]` là tập tất cả số nguyên từ `a` đến `b` (tính cả hai đầu).

Cần trả về **danh sách range nhỏ nhất (ít range nhất), theo thứ tự tăng dần**, sao cho:

- mỗi phần tử của `nums` nằm trong **đúng một** range;
- không có số nguyên `x` nào nằm trong range mà lại **không** có trong `nums`.

Cách in một range:

- `"a->b"` nếu `a != b`
- `"a"` nếu `a == b`

| `nums` | Kết quả | Giải thích |
|---|---|---|
| `[0,1,2,4,5,7]` | `["0->2","4->5","7"]` | 0,1,2 liền nhau; 4,5 liền nhau; 7 đứng một mình |
| `[0,2,3,4,6,8,9]` | `["0","2->4","6","8->9"]` | 0 lẻ loi; 2,3,4; 6 lẻ loi; 8,9 |
| `[]` | `[]` | mảng rỗng → không có range nào |

**Constraints**

- `0 <= nums.length <= 20`
- `-2^31 <= nums[i] <= 2^31 - 1`
- Mọi giá trị trong `nums` khác nhau, `nums` tăng dần.

Câu hỏi then chốt: **Khi nào hai số kề nhau trong mảng thuộc cùng một range?** → khi số sau đúng bằng số trước cộng 1.

In [1]:
nums1 = [0, 1, 2, 4, 5, 7]
nums2 = [0, 2, 3, 4, 6, 8, 9]
expected1 = ["0->2", "4->5", "7"]
expected2 = ["0", "2->4", "6", "8->9"]
print("Ví dụ 1:", nums1, "->", expected1)
print("Ví dụ 2:", nums2, "->", expected2)

Ví dụ 1: [0, 1, 2, 4, 5, 7] -> ['0->2', '4->5', '7']
Ví dụ 2: [0, 2, 3, 4, 6, 8, 9] -> ['0', '2->4', '6', '8->9']


## 2. Nền tảng Python cần dùng

### 2.1. List và index

`list` là dãy phần tử có thứ tự. Phần tử thứ `i` (đếm từ 0) đọc bằng `nums[i]`. Index hợp lệ là `0 .. len(nums) - 1`.
Đọc `nums[len(nums)]` sẽ gây `IndexError` — đây là lỗi kinh điển trong bài này khi ta "nhìn sang phần tử kế tiếp".

In [2]:
nums = [0, 1, 2, 4, 5, 7]
print("len =", len(nums))
print("nums[0] =", nums[0], "| nums[-1] =", nums[-1])
for i in range(len(nums) - 1):
    print(f"i={i}: nums[i]={nums[i]}, nums[i+1]={nums[i+1]}, liền nhau? {nums[i+1] == nums[i] + 1}")

len = 6
nums[0] = 0 | nums[-1] = 7
i=0: nums[i]=0, nums[i+1]=1, liền nhau? True
i=1: nums[i]=1, nums[i+1]=2, liền nhau? True
i=2: nums[i]=2, nums[i+1]=4, liền nhau? False
i=3: nums[i]=4, nums[i+1]=5, liền nhau? True
i=4: nums[i]=5, nums[i+1]=7, liền nhau? False


Nhận xét: vòng `range(len(nums) - 1)` dừng ở `i = len - 2`, nên `nums[i+1]` luôn hợp lệ. Cột "liền nhau?" chính là chỗ **cắt đoạn**: mỗi chữ `False` là ranh giới giữa hai range.

### 2.2. f-string để tạo chuỗi kết quả

`f"{a}->{b}"` chèn giá trị biến vào chuỗi. Với số âm vẫn đúng: `-3->-1`.

In [3]:
a, b = 4, 5
print(f"{a}->{b}")
a, b = -3, -1
print(f"{a}->{b}")
a = b = 7
print(f"{a}" if a == b else f"{a}->{b}")   # biểu thức điều kiện (ternary)

4->5
-3->-1
7


### 2.3. Biểu thức điều kiện một dòng

`X if cond else Y` trả về `X` khi `cond` đúng, ngược lại trả `Y`. Ta sẽ gói việc định dạng một range vào hàm nhỏ `fmt(a, b)` để không lặp code.

In [4]:
def fmt(a: int, b: int) -> str:
    """Định dạng range [a, b] theo yêu cầu đề."""
    return str(a) if a == b else f"{a}->{b}"

print(fmt(0, 2), fmt(7, 7), fmt(-5, -4))

0->2 7 -5->-4


### 2.4. `append` để gom kết quả

`result.append(x)` thêm `x` vào cuối list. Kết quả của bài là một list chuỗi, ta xây dần từ trái sang phải nên thứ tự tự động tăng dần.

In [5]:
result = []
result.append(fmt(0, 2))
result.append(fmt(4, 5))
result.append(fmt(7, 7))
print(result)

['0->2', '4->5', '7']


## 3. Cách trực giác — duyệt và hỏi "số này có nối tiếp số trước không?"

Ý tưởng người mới thường nghĩ:

1. Giữ "đoạn hiện tại" dưới dạng một list nhỏ `current`.
2. Với mỗi số `x`: nếu `current` rỗng hoặc `x == current[-1] + 1` thì thêm `x` vào `current`.
3. Ngược lại: đoạn hiện tại đã kết thúc → chốt nó vào kết quả, mở đoạn mới chỉ chứa `x`.
4. Hết vòng lặp, **đừng quên** chốt đoạn cuối cùng.

In [6]:
def summary_ranges_lists(nums: list[int]) -> list[str]:
    result = []
    current = []                     # đoạn liên tiếp đang xây
    for x in nums:
        if not current or x == current[-1] + 1:
            current.append(x)        # x nối tiếp được
        else:
            result.append(fmt(current[0], current[-1]))   # chốt đoạn cũ
            current = [x]            # mở đoạn mới
    if current:                      # chốt đoạn cuối
        result.append(fmt(current[0], current[-1]))
    return result

print(summary_ranges_lists(nums1))
print(summary_ranges_lists(nums2))
print(summary_ranges_lists([]))

['0->2', '4->5', '7']
['0', '2->4', '6', '8->9']
[]


### Nút thắt của cách trực giác

Cách này **đúng** và chạy `O(n)`, nhưng có điểm dư thừa:

- `current` lưu **toàn bộ** các số trong đoạn, trong khi để in ra ta chỉ cần **số đầu** và **số cuối**.
- Bộ nhớ phụ `O(n)` trong trường hợp cả mảng là một đoạn duy nhất (ví dụ `[1,2,3,...,20]`).

Câu hỏi cải tiến: *Có cần nhớ cả đoạn không, hay chỉ nhớ nơi đoạn bắt đầu?*

In [7]:
big = list(range(1, 21))
current_demo = []
for x in big:
    current_demo.append(x)
print("current giữ", len(current_demo), "phần tử, nhưng ta chỉ dùng", current_demo[0], "và", current_demo[-1])

current giữ 20 phần tử, nhưng ta chỉ dùng 1 và 20


## 4. Insight — chỉ cần nhớ chỉ số bắt đầu của đoạn

Một đoạn liên tiếp trong mảng đã sắp xếp là một **dải index liền nhau** `i, i+1, ..., j` mà mỗi bước giá trị tăng đúng 1.

Vậy ta dùng hai con trỏ:

- `i`: index **đầu** đoạn.
- `j`: chạy từ `i` sang phải **chừng nào** `nums[j+1] == nums[j] + 1`.

Khi `j` dừng, đoạn là `[nums[i], nums[j]]`. Sau đó đặt `i = j + 1` để bắt đầu đoạn tiếp theo.

Ví dụ `[0,1,2,4,5,7]`:

```text
i=0: j chạy 0→1→2 (vì 1=0+1, 2=1+1), dừng vì 4 != 3   → "0->2", i=3
i=3: j chạy 3→4 (5=4+1), dừng vì 7 != 6               → "4->5", i=5
i=5: j=5 là phần tử cuối, dừng                        → "7",    i=6 = n → hết
```

In [8]:
nums = [0, 1, 2, 4, 5, 7]
i = 0
j = i
while j + 1 < len(nums) and nums[j + 1] == nums[j] + 1:
    j += 1
print("Đoạn đầu tiên: i =", i, ", j =", j, "->", fmt(nums[i], nums[j]))

Đoạn đầu tiên: i = 0 , j = 2 -> 0->2


### Vì sao điều kiện `j + 1 < len(nums)` phải đứng **trước**?

Toán tử `and` trong Python **dừng sớm** (short-circuit): nếu vế trái `False` thì vế phải không được tính.
Nhờ vậy khi `j` là phần tử cuối, `nums[j + 1]` không bao giờ được đọc → không `IndexError`.

In [9]:
arr = [5]
j = 0
print("Kiểm tra an toàn:", j + 1 < len(arr) and arr[j + 1] == arr[j] + 1)
try:
    print(arr[j + 1] == arr[j] + 1 and j + 1 < len(arr))
except IndexError as e:
    print("Đảo thứ tự điều kiện -> IndexError:", e)

Kiểm tra an toàn: False
Đảo thứ tự điều kiện -> IndexError: list index out of range


## 5. Lời giải chính — hai con trỏ `i`, `j`

Ghép các mảnh lại:

1. `i = 0`, `result = []`.
2. Trong khi `i < n`:
   - đặt `j = i`;
   - kéo `j` sang phải chừng nào phần tử kế tiếp liền giá trị;
   - thêm `fmt(nums[i], nums[j])`;
   - nhảy `i = j + 1`.
3. Trả `result`.

Mảng rỗng: vòng `while i < 0` không chạy lần nào → trả `[]` tự nhiên, không cần if riêng.

In [10]:
def summary_ranges_two_pointers(nums: list[int]) -> list[str]:
    result = []
    n = len(nums)
    i = 0
    while i < n:
        j = i
        while j + 1 < n and nums[j + 1] == nums[j] + 1:
            j += 1
        result.append(fmt(nums[i], nums[j]))
        i = j + 1
    return result

print(summary_ranges_two_pointers(nums1))
print(summary_ranges_two_pointers(nums2))
print(summary_ranges_two_pointers([]))
print(summary_ranges_two_pointers([-2147483648, 2147483647]))

['0->2', '4->5', '7']
['0', '2->4', '6', '8->9']
[]
['-2147483648', '2147483647']


Hai vòng `while` lồng nhau **không** làm thuật toán thành `O(n²)`: con trỏ `j` chỉ tiến về phía trước, và `i` nhảy thẳng tới `j + 1`. Mỗi index được `j` đi qua đúng một lần → tổng số bước `O(n)`.

In [11]:
def count_steps(nums):
    steps = 0
    i, n = 0, len(nums)
    while i < n:
        j = i
        while j + 1 < n and nums[j + 1] == nums[j] + 1:
            j += 1
            steps += 1
        steps += 1          # bước chốt đoạn
        i = j + 1
    return steps

for arr in [list(range(20)), list(range(0, 40, 2)), nums2]:
    print(f"n={len(arr):2d}, số bước={count_steps(arr)}")

n=20, số bước=20
n=20, số bước=20
n= 7, số bước=7


Số bước luôn đúng bằng `n` — bằng chứng thực nghiệm cho `O(n)`.

## 6. Trace từng bước

Bảng trace cho `nums = [0,2,3,4,6,8,9]`:

| Vòng ngoài | `i` | `j` chạy qua | Lý do `j` dừng | Range thêm | `i` mới |
|---|---|---|---|---|---|
| 1 | 0 | 0 | `nums[1]=2 != 0+1` | `"0"` | 1 |
| 2 | 1 | 1→2→3 | `nums[4]=6 != 4+1` | `"2->4"` | 4 |
| 3 | 4 | 4 | `nums[5]=8 != 6+1` | `"6"` | 5 |
| 4 | 5 | 5→6 | `j+1 = 7 = n` (hết mảng) | `"8->9"` | 7 |

In [12]:
def trace_summary_ranges(nums: list[int]) -> list[str]:
    result = []
    n = len(nums)
    i = 0
    round_no = 1
    while i < n:
        j = i
        print(f"[vòng {round_no}] bắt đầu đoạn tại i={i} (giá trị {nums[i]})")
        while j + 1 < n and nums[j + 1] == nums[j] + 1:
            print(f"    nums[{j+1}]={nums[j+1]} == nums[{j}]+1={nums[j]+1} -> j tiến lên {j+1}")
            j += 1
        reason = "hết mảng" if j + 1 >= n else f"nums[{j+1}]={nums[j+1]} != {nums[j]+1}"
        result.append(fmt(nums[i], nums[j]))
        print(f"    dừng vì {reason}; thêm {result[-1]!r}; result={result}")
        i = j + 1
        round_no += 1
    return result

trace_summary_ranges([0, 2, 3, 4, 6, 8, 9])

[vòng 1] bắt đầu đoạn tại i=0 (giá trị 0)
    dừng vì nums[1]=2 != 1; thêm '0'; result=['0']
[vòng 2] bắt đầu đoạn tại i=1 (giá trị 2)
    nums[2]=3 == nums[1]+1=3 -> j tiến lên 2
    nums[3]=4 == nums[2]+1=4 -> j tiến lên 3
    dừng vì nums[4]=6 != 5; thêm '2->4'; result=['0', '2->4']
[vòng 3] bắt đầu đoạn tại i=4 (giá trị 6)
    dừng vì nums[5]=8 != 7; thêm '6'; result=['0', '2->4', '6']
[vòng 4] bắt đầu đoạn tại i=5 (giá trị 8)
    nums[6]=9 == nums[5]+1=9 -> j tiến lên 6
    dừng vì hết mảng; thêm '8->9'; result=['0', '2->4', '6', '8->9']


['0', '2->4', '6', '8->9']

Chú ý dòng cuối: đoạn `8->9` dừng vì **hết mảng**, không phải vì gặp số lệch. Đây là nhánh mà cách trực giác hay quên (phải "chốt đoạn cuối"). Trong cách hai con trỏ, việc chốt nằm ngay trong vòng ngoài nên không thể quên.

In [13]:
trace_summary_ranges([5])
print()
trace_summary_ranges([-3, -2, -1, 1])

[vòng 1] bắt đầu đoạn tại i=0 (giá trị 5)
    dừng vì hết mảng; thêm '5'; result=['5']

[vòng 1] bắt đầu đoạn tại i=0 (giá trị -3)
    nums[1]=-2 == nums[0]+1=-2 -> j tiến lên 1
    nums[2]=-1 == nums[1]+1=-1 -> j tiến lên 2
    dừng vì nums[3]=1 != 0; thêm '-3->-1'; result=['-3->-1']
[vòng 2] bắt đầu đoạn tại i=3 (giá trị 1)
    dừng vì hết mảng; thêm '1'; result=['-3->-1', '1']


['-3->-1', '1']

## 7. Cách 3 — Lưu `start`, duyệt một vòng `for`

Một cách viết khác rất phổ biến: chỉ một vòng `for`, giữ biến `start` là **giá trị** đầu đoạn.

Tại index `i`, đoạn hiện tại **kết thúc tại `i`** nếu:
- `i` là phần tử cuối, **hoặc**
- `nums[i + 1] != nums[i] + 1`.

Khi đoạn kết thúc: thêm `fmt(start, nums[i])` và đặt `start = nums[i + 1]` (nếu còn).

In [14]:
def summary_ranges_start(nums: list[int]) -> list[str]:
    result = []
    if not nums:
        return result
    start = nums[0]
    for i in range(len(nums)):
        is_last = (i == len(nums) - 1)
        if is_last or nums[i + 1] != nums[i] + 1:
            result.append(fmt(start, nums[i]))
            if not is_last:
                start = nums[i + 1]
    return result

print(summary_ranges_start(nums1))
print(summary_ranges_start(nums2))
print(summary_ranges_start([]))

['0->2', '4->5', '7']
['0', '2->4', '6', '8->9']
[]


## 8. Cách 4 — Mẹo "hiệu bất biến" `nums[i] - i` với `itertools.groupby`

Quan sát: trong một đoạn liên tiếp, khi index tăng 1 thì giá trị cũng tăng 1, nên **`nums[i] - i` không đổi** trong cả đoạn. Sang đoạn mới (giá trị nhảy nhiều hơn 1) thì hiệu này **tăng**.

In [15]:
nums = [0, 2, 3, 4, 6, 8, 9]
for i, x in enumerate(nums):
    print(f"i={i}, x={x}, x - i = {x - i}")

i=0, x=0, x - i = 0
i=1, x=2, x - i = 1
i=2, x=3, x - i = 1
i=3, x=4, x - i = 1
i=4, x=6, x - i = 2
i=5, x=8, x - i = 3
i=6, x=9, x - i = 3


`enumerate(nums)` trả lần lượt cặp `(index, giá trị)` — tiện hơn `range(len(...))` khi cần cả hai.

`itertools.groupby(iterable, key)` gom các phần tử **liên tiếp** có cùng `key` thành một nhóm. (Lưu ý: chỉ gom phần tử *kề nhau*, không gom toàn cục như dict.)

In [16]:
from itertools import groupby

for k, grp in groupby("aaabccdd"):
    print(k, list(grp))

a ['a', 'a', 'a']
b ['b']
c ['c', 'c']
d ['d', 'd']


In [17]:
def summary_ranges_groupby(nums: list[int]) -> list[str]:
    result = []
    for _, grp in groupby(enumerate(nums), key=lambda pair: pair[1] - pair[0]):
        block = [x for _, x in grp]          # các giá trị trong cùng nhóm
        result.append(fmt(block[0], block[-1]))
    return result

print(summary_ranges_groupby(nums1))
print(summary_ranges_groupby(nums2))
print(summary_ranges_groupby([]))

['0->2', '4->5', '7']
['0', '2->4', '6', '8->9']
[]


`lambda pair: pair[1] - pair[0]` là hàm vô danh một dòng: nhận `pair = (i, x)` và trả `x - i`. Cách này ngắn và đẹp, nhưng người mới nên hiểu chắc cách hai con trỏ trước, vì phỏng vấn thường chờ bạn giải thích bằng con trỏ.

## 9. Code nộp LeetCode

Bản chính (hai con trỏ), không print, đúng chữ ký:

In [18]:
class Solution:
    def summaryRanges(self, nums: list[int]) -> list[str]:
        result = []
        n = len(nums)
        i = 0
        while i < n:
            j = i
            while j + 1 < n and nums[j + 1] == nums[j] + 1:
                j += 1
            if i == j:
                result.append(str(nums[i]))
            else:
                result.append(f"{nums[i]}->{nums[j]}")
            i = j + 1
        return result

sol = Solution()
print(sol.summaryRanges([0, 1, 2, 4, 5, 7]))
print(sol.summaryRanges([0, 2, 3, 4, 6, 8, 9]))

['0->2', '4->5', '7']
['0', '2->4', '6', '8->9']


Bản dự phòng (groupby) — cũng nộp được:

```python
from itertools import groupby

class Solution:
    def summaryRanges(self, nums: list[int]) -> list[str]:
        res = []
        for _, g in groupby(enumerate(nums), key=lambda p: p[1] - p[0]):
            block = [x for _, x in g]
            res.append(str(block[0]) if len(block) == 1 else f"{block[0]}->{block[-1]}")
        return res
```

## 10. Vì sao thuật toán đúng?

**Invariant** của vòng ngoài: trước mỗi lần lặp, `result` chứa đúng các range tối đại của `nums[0 .. i-1]`, và `nums[i]` (nếu có) là **đầu** của một range mới (tức `i == 0` hoặc `nums[i] != nums[i-1] + 1`).

- Vòng trong kéo `j` xa nhất có thể, nên `[nums[i], nums[j]]` là đoạn liên tiếp **dài nhất** bắt đầu tại `i` → không thể gộp thêm phần tử phía sau (vì `nums[j+1] != nums[j] + 1` hoặc hết mảng).
- Mọi số nguyên giữa `nums[i]` và `nums[j]` đều có mặt (mỗi bước tăng đúng 1) → không có `x` thừa.
- Không thể gộp hai range liền kề vì giữa chúng có ít nhất một số nguyên bị thiếu → số range là **nhỏ nhất**.
- Mảng tăng dần nên các range sinh ra theo thứ tự tăng dần.

### Độ phức tạp

Gọi `n = len(nums)`.

| Cách | Time | Space phụ (không tính output) |
|---|---|---|
| List `current` | `O(n)` | `O(n)` (đoạn dài nhất) |
| Hai con trỏ `i/j` | `O(n)` — mỗi index được `j` đi qua 1 lần | `O(1)` |
| Biến `start` + `for` | `O(n)` | `O(1)` |
| `groupby(x - i)` | `O(n)` | `O(n)` cho `block` tạm |

Output có tối đa `n` chuỗi nên tổng bộ nhớ kể cả output là `O(n)` ở mọi cách.

## 11. Bộ test

Các nhóm test: ví dụ đề, mảng rỗng, một phần tử, toàn bộ là một đoạn, không có hai số nào liền nhau, số âm, giá trị biên `±2^31`, và đối chiếu 4 cách trên dữ liệu ngẫu nhiên.

In [19]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

sol = Solution()
check("ví dụ 1", sol.summaryRanges([0, 1, 2, 4, 5, 7]), ["0->2", "4->5", "7"])
check("ví dụ 2", sol.summaryRanges([0, 2, 3, 4, 6, 8, 9]), ["0", "2->4", "6", "8->9"])
check("rỗng", sol.summaryRanges([]), [])
check("một phần tử", sol.summaryRanges([-1]), ["-1"])
check("một đoạn", sol.summaryRanges([3, 4, 5, 6]), ["3->6"])
check("rời rạc", sol.summaryRanges([1, 3, 5]), ["1", "3", "5"])
check("số âm", sol.summaryRanges([-5, -4, -2, -1, 0, 3]), ["-5->-4", "-2->0", "3"])
check("biên int32", sol.summaryRanges([-2147483648, -2147483647, 2147483647]),
      ["-2147483648->-2147483647", "2147483647"])

PASS  ví dụ 1: ['0->2', '4->5', '7']
PASS  ví dụ 2: ['0', '2->4', '6', '8->9']
PASS  rỗng: []
PASS  một phần tử: ['-1']
PASS  một đoạn: ['3->6']
PASS  rời rạc: ['1', '3', '5']
PASS  số âm: ['-5->-4', '-2->0', '3']
PASS  biên int32: ['-2147483648->-2147483647', '2147483647']


In [20]:
import random

rng = random.Random(228)
solvers = [summary_ranges_lists, summary_ranges_two_pointers,
           summary_ranges_start, summary_ranges_groupby, Solution().summaryRanges]
for _ in range(500):
    size = rng.randint(0, 20)
    arr = sorted(rng.sample(range(-30, 30), size))
    outs = [f(arr) for f in solvers]
    assert all(o == outs[0] for o in outs), (arr, outs)
print("500 mảng ngẫu nhiên: 5 cài đặt cho kết quả giống hệt nhau — PASS")

500 mảng ngẫu nhiên: 5 cài đặt cho kết quả giống hệt nhau — PASS


Kiểm tra thêm **tính chất** của output thay vì chỉ so expected: ghép lại các range phải cho đúng `nums`.

In [21]:
def expand(ranges: list[str]) -> list[int]:
    out = []
    for r in ranges:
        if "->" in r:
            a, b = r.split("->")
            out.extend(range(int(a), int(b) + 1))
        else:
            out.append(int(r))
    return out

for _ in range(300):
    arr = sorted(rng.sample(range(-50, 50), rng.randint(0, 20)))
    assert expand(Solution().summaryRanges(arr)) == arr
print("expand(summaryRanges(nums)) == nums với 300 mảng — PASS")

expand(summaryRanges(nums)) == nums với 300 mảng — PASS


Hàm `expand` dùng `r.split("->")`. Với `"-5->-4"`, `split("->")` cho `["-5", "-4"]` — dấu trừ của số âm không bị nhầm vì ta tách theo cả cụm `"->"`.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên chốt đoạn cuối

In [22]:
def wrong_forget_last(nums):
    result = []
    if not nums:
        return result
    start = nums[0]
    for i in range(1, len(nums)):
        if nums[i] != nums[i - 1] + 1:
            result.append(fmt(start, nums[i - 1]))
            start = nums[i]
    return result                       # thiếu: chốt đoạn [start, nums[-1]]

print("sai:", wrong_forget_last([0, 1, 2, 4, 5, 7]))

sai: ['0->2', '4->5']


**Vì sao sai:** đoạn chỉ được chốt khi gặp "chỗ gãy", nhưng đoạn cuối không có chỗ gãy phía sau. **Sửa:** sau vòng lặp thêm `result.append(fmt(start, nums[-1]))`.

In [23]:
def fixed_forget_last(nums):
    result = []
    if not nums:
        return result
    start = nums[0]
    for i in range(1, len(nums)):
        if nums[i] != nums[i - 1] + 1:
            result.append(fmt(start, nums[i - 1]))
            start = nums[i]
    result.append(fmt(start, nums[-1]))  # chốt đoạn cuối
    return result

print("đúng:", fixed_forget_last([0, 1, 2, 4, 5, 7]))

đúng: ['0->2', '4->5', '7']


### Lỗi 2: Đọc `nums[0]` khi mảng rỗng

`start = nums[0]` với `nums = []` → `IndexError`. Constraints cho phép `nums.length = 0`, nên phải có `if not nums: return []` (hoặc dùng cách hai con trỏ tự xử lý).

In [24]:
try:
    start = [][0]
except IndexError as e:
    print("IndexError:", e)
print("fixed_forget_last([]) =", fixed_forget_last([]))

IndexError: list index out of range
fixed_forget_last([]) = []


### Lỗi 3: Kiểm tra "liền nhau" bằng hiệu `<= 1` hoặc so với `nums[i] + 1` theo hướng sai

Viết `nums[i] - nums[i-1] <= 1` vẫn đúng ở bài này vì không có phần tử trùng, nhưng nếu đề biến thể cho phép trùng, nó sẽ gộp sai. Rõ ràng nhất là so sánh chính xác `== nums[i-1] + 1`.

### Lỗi 4: In range sai định dạng

Viết `f"{a}-{b}"` hoặc `f"{a} -> {b}"` (có dấu cách) đều bị LeetCode chấm sai. Đúng: `f"{a}->{b}"` và chỉ `"a"` khi `a == b`.

In [25]:
print(repr(f"{4} -> {5}"), "sai  |", repr(f"{4}->{5}"), "đúng")

'4 -> 5' sai  | '4->5' đúng


### Lỗi 5 (ngôn ngữ khác): tràn số khi tính `nums[i] + 1`

Ở C++/Java, `nums[i] = 2^31 - 1` thì `nums[i] + 1` tràn `int`. Python có số nguyên không giới hạn nên an toàn, nhưng khi phỏng vấn nên nói: "dùng `nums[i+1] - nums[i] == 1` với kiểu `long`" hoặc so sánh `nums[i+1] != nums[i] + 1` bằng `long`.

## 13. Mẹo nhận dạng pattern

- Đề có **mảng đã sắp xếp** + yêu cầu **nhóm các đoạn liên tiếp / gom khối** → nghĩ ngay tới **hai con trỏ đầu-cuối đoạn** (pattern "group consecutive" hay "run-length").
- Dấu hiệu "mỗi phần tử thuộc đúng một nhóm, các nhóm theo thứ tự" → một lần duyệt trái sang phải là đủ.
- Khi so sánh với phần tử kế tiếp, luôn tự hỏi: **phần tử cuối được xử lý ở đâu?**
- Mẹo `x - i` bất biến dùng được cho mọi bài "dãy tăng đúng 1 mỗi bước".

## 14. Biến thể

### Biến thể A — Missing Ranges (LeetCode 163)
Cho `lower`, `upper` và `nums`, liệt kê các **khoảng thiếu**. Đây là "phần bù" của Summary Ranges.

### Biến thể B — Run-length encoding
Gom các ký tự giống nhau liên tiếp: `"aaabcc"` → `"a3b1c2"`. Cùng khung hai con trỏ, chỉ đổi điều kiện "liền nhau" thành "bằng nhau".

In [26]:
def run_length(s: str) -> str:
    out = []
    i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        out.append(f"{s[i]}{j - i + 1}")
        i = j + 1
    return "".join(out)

print(run_length("aaabcc"), run_length("z"), repr(run_length("")))

a3b1c2 z1 ''


`"".join(list_chuoi)` nối các chuỗi trong list lại, dùng chuỗi rỗng làm dấu nối.

### Biến thể C — Trả về range dạng cặp số
Thay vì chuỗi, trả `[[0,2],[4,5],[7,7]]`. Chỉ cần đổi dòng `append`.

In [27]:
def ranges_as_pairs(nums):
    res, i = [], 0
    while i < len(nums):
        j = i
        while j + 1 < len(nums) and nums[j + 1] == nums[j] + 1:
            j += 1
        res.append([nums[i], nums[j]])
        i = j + 1
    return res

print(ranges_as_pairs([0, 1, 2, 4, 5, 7]))

[[0, 2], [4, 5], [7, 7]]


## 15. Bài tập tự luyện (TODO)

Làm theo thứ tự. **Đừng xem đáp án ở cuối notebook trước khi tự thử.**

### Bài 1 — Tính tay
Không chạy code, hãy viết ra kết quả `summaryRanges([-1, 0, 1, 3, 4, 10])`. Ghi lại `i`, `j` ở mỗi vòng ngoài.

### Bài 2 — Đếm số range
Viết `count_ranges(nums)` trả về **số lượng** range mà không tạo chuỗi.

In [28]:
def exercise_count_ranges(nums: list[int]) -> int:
    # TODO: đếm số "chỗ bắt đầu đoạn": i == 0 hoặc nums[i] != nums[i-1] + 1
    pass

# Test sau khi làm xong:
# assert exercise_count_ranges([0, 1, 2, 4, 5, 7]) == 3
# assert exercise_count_ranges([]) == 0

### Bài 3 — Tự viết lại lời giải hai con trỏ

In [29]:
def exercise_summary_ranges(nums: list[int]) -> list[str]:
    # TODO 1: result = [], i = 0
    # TODO 2: while i < len(nums): đặt j = i, kéo j khi nums[j+1] == nums[j] + 1
    # TODO 3: thêm "a" hoặc "a->b"; nhảy i = j + 1
    pass

# assert exercise_summary_ranges([0, 2, 3, 4, 6, 8, 9]) == ["0", "2->4", "6", "8->9"]

### Bài 4 — Missing Ranges
Cho `nums` tăng dần, không trùng, nằm trong `[lower, upper]`. Trả về list các cặp `[a, b]` là những đoạn số **không có** trong `nums`.

Ví dụ: `nums=[0,1,3,50,75], lower=0, upper=99` → `[[2,2],[4,49],[51,74],[76,99]]`.

In [30]:
def exercise_missing_ranges(nums: list[int], lower: int, upper: int) -> list[list[int]]:
    # TODO: duyệt với biến prev = lower - 1, thêm "lính canh" upper + 1 ở cuối
    pass

# assert exercise_missing_ranges([0, 1, 3, 50, 75], 0, 99) == [[2, 2], [4, 49], [51, 74], [76, 99]]
# assert exercise_missing_ranges([], 1, 1) == [[1, 1]]

### Bài 5 — Đoạn liên tiếp dài nhất
Trả về độ dài của range dài nhất. Ví dụ `[0,2,3,4,6,8,9]` → `3`.

In [31]:
def exercise_longest_run(nums: list[int]) -> int:
    # TODO: dùng hai con trỏ, theo dõi max(j - i + 1)
    pass

# assert exercise_longest_run([0, 2, 3, 4, 6, 8, 9]) == 3
# assert exercise_longest_run([]) == 0

### Bài 6 — Mini challenge: mảng chưa sắp xếp, có trùng
Cho `nums` **bất kỳ** (có thể trùng, chưa sắp xếp). Trả về summary ranges của **tập giá trị**. Ví dụ `[5, 1, 2, 2, 3, 9]` → `["1->3", "5", "9"]`.

In [32]:
def exercise_summary_unsorted(nums: list[int]) -> list[str]:
    # TODO: loại trùng + sắp xếp, rồi dùng lại thuật toán
    pass

# assert exercise_summary_unsorted([5, 1, 2, 2, 3, 9]) == ["1->3", "5", "9"]

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Tìm các chỗ gãy (hiệu ≠ 1) | Gãy giữa 1 và 3, giữa 4 và 10 | `["-1->1","3->4","10"]` có 3 vòng ngoài |
| 2 | Số range = số điểm bắt đầu đoạn | Duyệt `i` từ 0, đếm điều kiện | `sum(1 for i in range(n) if i == 0 or nums[i] != nums[i-1]+1)` |
| 3 | Chép lại khung trong mục 5 | `while i < n` / `while j+1 < n and ...` | Nhớ `i = j + 1` cuối vòng |
| 4 | Khoảng thiếu nằm giữa hai số liên tiếp | `prev = lower - 1`; duyệt `nums + [upper + 1]` | nếu `x - prev >= 2` thêm `[prev+1, x-1]` |
| 5 | Mỗi vòng ngoài cho một đoạn | độ dài đoạn = `j - i + 1` | `best = max(best, j - i + 1)` |
| 6 | Biến về đúng giả thiết của đề | `sorted(set(nums))` | Gọi lại `Solution().summaryRanges(...)` |

## 17. Đáp án tham khảo

Bài 1 (tính tay): vòng 1 `i=0, j: 0→1→2` → `"-1->1"`; vòng 2 `i=3, j: 3→4` → `"3->4"`; vòng 3 `i=5, j=5` → `"10"`. Kết quả `["-1->1", "3->4", "10"]`.

In [33]:
def answer_count_ranges(nums):
    return sum(1 for i in range(len(nums)) if i == 0 or nums[i] != nums[i - 1] + 1)

def answer_summary_ranges(nums):
    result, i, n = [], 0, len(nums)
    while i < n:
        j = i
        while j + 1 < n and nums[j + 1] == nums[j] + 1:
            j += 1
        result.append(str(nums[i]) if i == j else f"{nums[i]}->{nums[j]}")
        i = j + 1
    return result

def answer_missing_ranges(nums, lower, upper):
    res = []
    prev = lower - 1
    for x in nums + [upper + 1]:          # lính canh upper + 1
        if x - prev >= 2:
            res.append([prev + 1, x - 1])
        prev = x
    return res

def answer_longest_run(nums):
    best, i, n = 0, 0, len(nums)
    while i < n:
        j = i
        while j + 1 < n and nums[j + 1] == nums[j] + 1:
            j += 1
        best = max(best, j - i + 1)
        i = j + 1
    return best

def answer_summary_unsorted(nums):
    return answer_summary_ranges(sorted(set(nums)))

assert answer_summary_ranges([-1, 0, 1, 3, 4, 10]) == ["-1->1", "3->4", "10"]
assert answer_count_ranges([0, 1, 2, 4, 5, 7]) == 3
assert answer_count_ranges([]) == 0
assert answer_summary_ranges([0, 2, 3, 4, 6, 8, 9]) == ["0", "2->4", "6", "8->9"]
assert answer_missing_ranges([0, 1, 3, 50, 75], 0, 99) == [[2, 2], [4, 49], [51, 74], [76, 99]]
assert answer_missing_ranges([], 1, 1) == [[1, 1]]
assert answer_missing_ranges([1], 1, 1) == []
assert answer_longest_run([0, 2, 3, 4, 6, 8, 9]) == 3
assert answer_longest_run([]) == 0
assert answer_summary_unsorted([5, 1, 2, 2, 3, 9]) == ["1->3", "5", "9"]
for _ in range(200):
    arr = sorted(rng.sample(range(-20, 20), rng.randint(0, 15)))
    assert answer_count_ranges(arr) == len(Solution().summaryRanges(arr))
print("Đáp án tham khảo: tất cả test PASS")

Đáp án tham khảo: tất cả test PASS


## 18. Template nhớ nhanh + checklist submit

```python
i = 0
while i < n:
    j = i
    while j + 1 < n and <nums[j+1] nối tiếp nums[j]>:
        j += 1
    <xử lý khối nums[i..j]>
    i = j + 1
```

**Checklist trước khi submit**

- [ ] Mảng rỗng trả `[]`, không đọc `nums[0]`.
- [ ] Điều kiện `j + 1 < n` đứng **trước** `nums[j + 1]`.
- [ ] Đoạn cuối được chốt (không bị bỏ sót).
- [ ] Định dạng đúng `"a->b"` (không dấu cách) và `"a"` khi một phần tử.
- [ ] Số âm in đúng, ví dụ `"-2->0"`.
- [ ] Nói được vì sao hai vòng `while` lồng nhau vẫn là `O(n)`.